# GDELT Article Body Scraper

This notebook scrapes body text for articles that are in the `articles` DB table
but have `body IS NULL`. It updates each row in-place via `UPDATE` — no table
replacements, no CSVs as intermediate format.

**Resume-capable:** re-running this notebook restarts only from unscraped rows.
The DB commit every 50 articles is the checkpoint — if interrupted, all committed
articles have `body != NULL` and are skipped on the next run.

**Pipeline:**
1. Load `body IS NULL` articles from DB
2. Scrape → clean → validate inline per article
3. `UPDATE articles SET body=?, body_valid=? WHERE id=?` — commit every 50
4. After the run: export full articles table to CSV as backup

### Necessary imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import html
import os

### Load pending articles from DB

In [3]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
df_todo = pd.read_sql("""
    SELECT id, title, url, domain
    FROM articles
    WHERE body IS NULL
""", conn)
conn.close()
print(f"Articles pending scrape: {len(df_todo)}")

Articles pending scrape: 5970


### Scraping and validation functions

Defines the three helpers used per article:

- **`scrape_article`** discards non-content HTML (`script`, `style`, `nav`, `footer`, `header`, `aside`) and returns the first 3 substantive paragraphs (>50 characters).
- **`clean_body`** normalizes encoding, strips control characters, and collapses whitespace.
- **`is_valid_body`** sets the `body_valid` flag: a body must be at least 400 characters, contain at least one oil/market keyword (`ALLOW_KEYWORDS`), not read as a corporate press release (`PRESS_RELEASE_PHRASES`), and not be mostly uppercase (a boilerplate signal). This filter is what separates usable article text from navigation dumps and PR noise downstream.

In [4]:
def scrape_article(url, timeout=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        soup = BeautifulSoup(response.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 50]
        return ' '.join(paragraphs[:3])
    except Exception as e:
        return f"ERROR: {str(e)[:100]}"


ALLOW_KEYWORDS = [
    'oil', 'crude', 'wti', 'brent', 'opec', 'barrel', 'petroleum',
    'energy', 'price', 'market', 'supply', 'demand', 'production',
    'refinery', 'inventory', 'futures', 'trading', 'commodity',
    'inflation', 'fed', 'dollar', 'interest rate', 'gdp'
]

PRESS_RELEASE_PHRASES = [
    "newsfile corp", "tsxv:", "otcqx:", "tsx:", "nyse:", "non-ifrs",
    "international financial reporting standards", "this press release",
    "forward-looking statements", "is pleased to announce", "is pleased to provide",
]


def clean_body(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def is_valid_body(text):
    if not isinstance(text, str) or len(text) < 400 or text.startswith('ERROR'):
        return False
    text_lower = text.lower()
    if not any(kw in text_lower for kw in ALLOW_KEYWORDS):
        return False
    if any(phrase in text_lower for phrase in PRESS_RELEASE_PHRASES):
        return False
    words = text.split()
    if len(words) > 10:
        cap_ratio = sum(1 for w in words if w.isupper() and len(w) > 2) / len(words)
        if cap_ratio > 0.3:
            return False
    return True


print("Functions ready: scrape_article, clean_body, is_valid_body")

Functions ready: scrape_article, clean_body, is_valid_body


### Test scraping on 5 sample articles

In [6]:
sample = df_todo.sample(5, random_state=42).reset_index(drop=True)

for _, row in sample.iterrows():
    print(f"\nDomain: {row['domain']}")
    print(f"Title:  {row['title'][:80]}")
    body = scrape_article(row['url'])
    body = clean_body(body)
    print(f"Valid:  {is_valid_body(body)}")
    print(f"Body ({len(body)} chars): {body[:200]}")
    print("-" * 60)


Domain: thehindubusinessline.com
Title:  Oil traders rush to hedge Iran risk after wild start to year
Valid:  True
Body (661 chars): The oil market is in the middle of its strongest start to a year since 2022 as supply shocks and sanctions confound expectations of a glut. Now traders are racing to cover themselves against the prosp
------------------------------------------------------------

Domain: newindianexpress.com
Title:  Fuel price hike : Supply shock set to outlast Strait disruption
Valid:  True
Body (513 chars): US President Donald Trump's 'little excursion' to the Middle East has landed the world in its biggest energy crisis in history. As the Iran war crossed its miserable two-month mark, brent crude future
------------------------------------------------------------

Domain: finance.yahoo.com
Title:  Stocks Pressured by Tech Weakness and Iran Tensions
Valid:  True
Body (1087 chars): The S&P 500 Index ($SPX) (SPY) today is down -0.75%, the Dow Jones Industrials Index ($DOW

### Scrape pending articles

Iterates over all `body IS NULL` articles. Each article is scraped, cleaned, and
validated inline, then written to the DB immediately via `UPDATE`.

**Checkpoint:** the DB is committed every 50 articles. If interrupted, re-running
restarts from where it left off — only `body IS NULL` rows are scraped.

**Expected time:** ~1.5h for 8,918 articles at 0.5s sleep.

In [5]:
BATCH_SIZE = 25
SLEEP_SECONDS = 0.5

conn = sqlite3.connect("../01_data/wti_thesis.db")
cursor = conn.cursor()

df_todo = pd.read_sql("SELECT id, title, url, domain FROM articles WHERE body IS NULL", conn)
total = len(df_todo)
print(f"Articles to scrape: {total}")

n_done = 0
n_valid = 0
n_failed = 0

for _, row in df_todo.iterrows():
    raw = scrape_article(row['url'])
    body = clean_body(raw)
    valid = is_valid_body(body)

    cursor.execute(
        "UPDATE articles SET body = ?, body_valid = ? WHERE id = ?",
        (body if body else None, int(valid), int(row['id']))
    )

    n_done += 1
    if valid:
        n_valid += 1
    if raw.startswith('ERROR'):
        n_failed += 1

    if n_done % BATCH_SIZE == 0:
        conn.commit()
        print(f"[{n_done}/{total}] valid: {n_valid} | errors: {n_failed}")

    time.sleep(SLEEP_SECONDS)

conn.commit()
print(f"\nDone. {n_done} scraped | {n_valid} valid ({n_valid/max(n_done,1)*100:.1f}%) | {n_failed} errors")
conn.close()

Articles to scrape: 5970
[25/5970] valid: 0 | errors: 0
[50/5970] valid: 0 | errors: 0
[75/5970] valid: 0 | errors: 0
[100/5970] valid: 0 | errors: 0
[125/5970] valid: 0 | errors: 0
[150/5970] valid: 0 | errors: 0
[175/5970] valid: 1 | errors: 0
[200/5970] valid: 1 | errors: 0
[225/5970] valid: 1 | errors: 0
[250/5970] valid: 1 | errors: 0
[275/5970] valid: 1 | errors: 0
[300/5970] valid: 1 | errors: 0
[325/5970] valid: 1 | errors: 0
[350/5970] valid: 1 | errors: 0
[375/5970] valid: 1 | errors: 0
[400/5970] valid: 1 | errors: 0
[425/5970] valid: 2 | errors: 0
[450/5970] valid: 2 | errors: 0
[475/5970] valid: 2 | errors: 0
[500/5970] valid: 2 | errors: 0
[525/5970] valid: 2 | errors: 0
[550/5970] valid: 2 | errors: 0
[575/5970] valid: 2 | errors: 0
[600/5970] valid: 2 | errors: 0
[625/5970] valid: 2 | errors: 0
[650/5970] valid: 2 | errors: 0
[675/5970] valid: 2 | errors: 0
[700/5970] valid: 2 | errors: 0
[725/5970] valid: 2 | errors: 0
[750/5970] valid: 3 | errors: 0
[775/5970] valid: 

### Verify results

In [6]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

stats = pd.read_sql("""
    SELECT
        SUM(CASE WHEN body IS NULL THEN 1 ELSE 0 END)          as still_pending,
        SUM(CASE WHEN body_valid = 1 THEN 1 ELSE 0 END)        as valid,
        SUM(CASE WHEN body_valid = 0 THEN 1 ELSE 0 END)        as invalid,
        COUNT(*)                                                 as total
    FROM articles
""", conn)
print(stats)

sample = pd.read_sql("""
    SELECT domain, substr(body, 1, 150) as body_preview
    FROM articles WHERE body_valid = 1
    ORDER BY RANDOM() LIMIT 5
""", conn)
print(sample.to_string())
conn.close()

   still_pending  valid  invalid  total
0           2749  13550     9245  22795
                         domain                                                                                                                                            body_preview
0  economictimes.indiatimes.com  OPEC+'s V8 group announced a 206,000 bpd production increase, citing market fundamentals despite Iran's regional attacks. Analysts warn this is insuff
1              businesstoday.in  Russian crude has become even cheaper for Indian refiners, with discounts widening to $3–$4 per barrel just as the Donald Trump administration imposes
2                     yahoo.com  WASHINGTON (AP) — The United States on Friday imposed sanctions on a fleet of nine ships and their owners accused of transporting hundreds of millions
3             investorplace.com  Oil prices continue to be volatile and hard to predict, which is why finding undervaluedenergy stockscan be hard, too. After rising steadily for most 


### Export CSV backup

In [7]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
df_out = pd.read_sql(
    "SELECT id, title, datetime, domain, url, body, body_valid FROM articles WHERE body IS NOT NULL",
    conn
)
conn.close()

out_path = "../01_data/raw/news/gdelt_wti_with_body_raw.csv"
os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Backup CSV saved: {out_path} ({len(df_out)} articles)")

Backup CSV saved: ../01_data/raw/news/gdelt_wti_with_body_raw.csv (20046 articles)
